# 🩺 Healthcare RAG System — Clinical Decision Support
### Powered by Google Gemini + LangChain + FAISS

This notebook builds a complete **Retrieval-Augmented Generation (RAG)** pipeline for clinical decision support.

**Architecture:**
```
Medical PDFs / TXTs
       ↓
  Document Loader
       ↓
  Text Splitter (512 tokens)
       ↓
  Google Embeddings → FAISS Vector Store
       ↓
  Similarity Search (Top-5 chunks)
       ↓
  Clinical Prompt + Gemini LLM
       ↓
  Answer + Source Documents
```

---
> ⚠️ **Disclaimer:** For clinical decision *support* only. Always validate with a licensed clinician.

## Step 1 — Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai \
    google-generativeai faiss-cpu sentence-transformers \
    pypdf python-dotenv streamlit

## Step 2 — Set Your Gemini API Key

In [ ]:
import os

# ✅ Paste your Gemini API key here
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY_HERE"

# OR load from .env file:
# from dotenv import load_dotenv
# load_dotenv()  # expects GOOGLE_API_KEY in .env file

print("✅ API Key set!")

## Step 3 — Import Libraries

In [ ]:
import os
from pathlib import Path
from IPython.display import Markdown, display

# LangChain
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Google Gemini
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

print("✅ All libraries imported successfully!")

## Step 4 — Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────
DATA_DIR        = Path("data")         # Put your medical PDFs/TXTs here
VECTORSTORE_DIR = Path("vectorstore")  # FAISS index saved here
DATA_DIR.mkdir(exist_ok=True)
VECTORSTORE_DIR.mkdir(exist_ok=True)

# ── Chunking ───────────────────────────────────────────
CHUNK_SIZE    = 512   # characters per chunk
CHUNK_OVERLAP = 64    # overlap between consecutive chunks

# ── Retrieval ──────────────────────────────────────────
TOP_K = 5             # number of chunks retrieved per query

# ── Gemini Models ──────────────────────────────────────
EMBED_MODEL = "models/embedding-001"     # Google embedding model
LLM_MODEL   = "gemini-1.5-flash"         # Gemini LLM (fast + free tier)
# LLM_MODEL = "gemini-1.5-pro"           # Uncomment for higher quality

print(f"✅ Config ready | Chunks: {CHUNK_SIZE} tokens | Top-K: {TOP_K} | LLM: {LLM_MODEL}")

## Step 5 — Create Sample Clinical Data
*(Skip this cell if you already have your own PDFs in the `data/` folder)*

In [ ]:
sample_text = """
CLINICAL GUIDELINES SUMMARY
============================

COMMUNITY-ACQUIRED PNEUMONIA (CAP)
-----------------------------------
First-line treatment for outpatient CAP in adults with no comorbidities:
- Amoxicillin 500 mg orally three times daily for 5 days (preferred)
- Doxycycline 100 mg orally twice daily for 5 days (alternative)
- Azithromycin 500 mg on Day 1, then 250 mg Days 2-5 (low-resistance areas only)

Inpatient non-ICU CAP:
- Beta-lactam (ceftriaxone/ampicillin-sulbactam) PLUS macrolide, OR
- Respiratory fluoroquinolone (levofloxacin or moxifloxacin) monotherapy

ICU CAP: Beta-lactam + azithromycin OR beta-lactam + respiratory fluoroquinolone.

SEPSIS (Sepsis-3 Criteria)
-----------------------------------
Definition: Life-threatening organ dysfunction due to dysregulated host response to infection.
Diagnostic criteria: Suspected infection + SOFA score increase >= 2 points.

Septic shock criteria:
- Vasopressor needed to maintain MAP >= 65 mmHg, AND
- Serum lactate > 2 mmol/L despite adequate fluids

Hour-1 Bundle:
1. Measure lactate (remeasure if > 2 mmol/L)
2. Blood cultures before antibiotics
3. Broad-spectrum antibiotics within 1 hour
4. 30 mL/kg IV crystalloid for hypotension or lactate >= 4 mmol/L
5. Vasopressors (norepinephrine first-line) if hypotension persists

STROKE — THROMBOLYTIC THERAPY (IV tPA)
-----------------------------------
Indication: Ischemic stroke, treatable within 3-4.5 hours of symptom onset.

Absolute Contraindications:
- Hemorrhagic stroke / intracranial hemorrhage on CT
- Stroke or significant head trauma in prior 3 months
- Intracranial neoplasm, AVM, or aneurysm
- Recent intracranial/spinal surgery
- Active internal bleeding
- Uncontrolled HTN: SBP > 185 mmHg or DBP > 110 mmHg
- Platelet count < 100,000/mm3
- INR > 1.7 or PT > 15 sec on anticoagulants
- Blood glucose < 50 mg/dL

H. PYLORI ERADICATION
-----------------------------------
Preferred first-line (Bismuth Quadruple Therapy):
- Bismuth subsalicylate 525 mg QID + Metronidazole 250 mg QID
+ Tetracycline 500 mg QID + PPI twice daily x 10-14 days

Alternative (Clarithromycin Triple — only if local resistance < 15%):
- Clarithromycin 500 mg + Amoxicillin 1 g + PPI — twice daily x 14 days

Test for eradication: Urea breath test or stool antigen test >= 4 weeks post-therapy.

DIABETES TYPE 2 — MANAGEMENT
-----------------------------------
HbA1c targets: < 7% for most adults; < 8% for elderly with comorbidities.
First-line: Metformin (if eGFR >= 30 mL/min/1.73 m2)
With ASCVD/high CV risk: Add GLP-1 RA or SGLT-2 inhibitor
With heart failure: Prefer SGLT-2 inhibitor
With CKD: Prefer SGLT-2 inhibitor (eGFR >= 20) or GLP-1 RA

Hypoglycemia — 15-15 Rule:
- 15 g fast-acting carbs; recheck glucose in 15 minutes
- Severe: Glucagon IM/SC/IN or IV dextrose 25 g

HYPERTENSION MANAGEMENT
-----------------------------------
BP targets: < 130/80 mmHg for most adults (ACC/AHA 2017)
First-line agents:
- Thiazide diuretics (chlorthalidone preferred)
- ACE inhibitors or ARBs (especially with DM or CKD)
- Calcium channel blockers (especially in elderly or Black patients)
Compelling indications:
- Post-MI: Beta-blocker + ACE inhibitor
- Heart failure with reduced EF: ACE inhibitor + beta-blocker + MRA
- CKD with proteinuria: ACE inhibitor or ARB
"""

(DATA_DIR / "clinical_guidelines.txt").write_text(sample_text)
print("✅ Sample clinical guidelines saved to data/clinical_guidelines.txt")

## Step 6 — Load & Split Documents

In [ ]:
def load_documents(data_dir: Path):
    """Load all PDF and TXT files from the data directory."""
    docs = []

    # Load PDFs
    pdf_files = list(data_dir.glob("**/*.pdf"))
    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        docs.extend(loader.load())
        print(f"  📄 Loaded PDF: {pdf_path.name}")

    # Load TXT files
    txt_files = list(data_dir.glob("**/*.txt"))
    for txt_path in txt_files:
        loader = TextLoader(str(txt_path), encoding="utf-8")
        docs.extend(loader.load())
        print(f"  📝 Loaded TXT: {txt_path.name}")

    print(f"\n✅ Total documents loaded: {len(docs)}")
    return docs


def split_documents(docs):
    """Split documents into overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    print(f"✅ Split into {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
    return chunks


# Run
docs   = load_documents(DATA_DIR)
chunks = split_documents(docs)

# Preview a chunk
print("\n--- Sample Chunk ---")
print(chunks[0].page_content)

## Step 7 — Create Google Embeddings & Build FAISS Index

In [ ]:
def build_vectorstore(chunks, save_dir: Path):
    """Embed chunks with Google embeddings and save FAISS index."""
    print("🔄 Creating Google embeddings... (this may take a moment)")
    
    embeddings = GoogleGenerativeAIEmbeddings(
        model=EMBED_MODEL,
        google_api_key=os.environ["GOOGLE_API_KEY"],
    )

    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(str(save_dir))
    print(f"✅ FAISS index built and saved to '{save_dir}/'")
    return vectorstore


def load_vectorstore(save_dir: Path):
    """Load existing FAISS index from disk."""
    embeddings = GoogleGenerativeAIEmbeddings(
        model=EMBED_MODEL,
        google_api_key=os.environ["GOOGLE_API_KEY"],
    )
    vs = FAISS.load_local(
        str(save_dir),
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print(f"✅ FAISS index loaded from '{save_dir}/'")
    return vs


# Build the index
vectorstore = build_vectorstore(chunks, VECTORSTORE_DIR)

## Step 8 — Initialize Gemini LLM

In [ ]:
llm = ChatGoogleGenerativeAI(
    model=LLM_MODEL,
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0,          # Deterministic — critical for clinical use
    convert_system_message_to_human=True,
)

print(f"✅ Gemini LLM ready: {LLM_MODEL}")

## Step 9 — Clinical Prompt Template

In [ ]:
CLINICAL_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert clinical decision support assistant.
Your role is to help clinicians by providing evidence-based information
strictly from the provided medical context.

IMPORTANT RULES:
1. Answer ONLY from the context provided. Do not use outside knowledge.
2. If the answer is not in the context, say: "I cannot find sufficient
   clinical evidence in the provided documents for this question."
3. Never fabricate drug dosages, diagnoses, or treatment protocols.
4. Structure your answer clearly with bullet points where appropriate.
5. Always recommend consulting a licensed clinician for final decisions.

Context:
{context}

Clinical Question:
{question}

Evidence-Based Clinical Answer:"""
)

print("✅ Clinical prompt template created")

## Step 10 — Build the RAG Chain

In [ ]:
def build_rag_chain(vectorstore, llm):
    """Assemble the full RAG retrieval chain."""
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": TOP_K},
    )
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": CLINICAL_PROMPT},
    )
    return chain


rag_chain = build_rag_chain(vectorstore, llm)
print("✅ RAG chain assembled and ready!")

## Step 11 — Query Function

In [ ]:
def ask_clinical_question(question: str, chain):
    """Run a clinical question through the RAG pipeline and display results."""
    print(f"\n{'='*60}")
    print(f"🔍 QUESTION: {question}")
    print('='*60)

    result = chain({"query": question})
    answer = result["result"]
    sources = result["source_documents"]

    print("\n🩺 CLINICAL ANSWER:")
    print("-" * 60)
    print(answer)

    print("\n📚 SOURCE DOCUMENTS USED:")
    print("-" * 60)
    for i, doc in enumerate(sources, 1):
        src  = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "N/A")
        excerpt = doc.page_content[:200].replace("\n", " ")
        print(f"[{i}] {Path(src).name} | Page: {page}")
        print(f"    → {excerpt}...")

    print('='*60)
    return answer, sources


print("✅ Query function ready!")

## Step 12 — Test Queries 🧪

In [ ]:
# Query 1: CAP Treatment
answer, sources = ask_clinical_question(
    "What is the first-line treatment for community-acquired pneumonia in outpatient adults?",
    rag_chain
)

In [ ]:
# Query 2: Sepsis
answer, sources = ask_clinical_question(
    "What are the diagnostic criteria for sepsis and what is the hour-1 bundle?",
    rag_chain
)

In [ ]:
# Query 3: Stroke contraindications
answer, sources = ask_clinical_question(
    "What are the absolute contraindications for IV tPA in ischemic stroke?",
    rag_chain
)

In [ ]:
# Query 4: Diabetes
answer, sources = ask_clinical_question(
    "What is the recommended HbA1c target and first-line medication for Type 2 Diabetes?",
    rag_chain
)

In [ ]:
# ✍️ Try your own question!
your_question = "What is the treatment for H. pylori?"  # ← Change this
answer, sources = ask_clinical_question(your_question, rag_chain)

## Step 13 — Add Your Own PDF Documents

In [ ]:
# To add your own medical PDFs:
# 1. Upload them to the 'data/' folder
# 2. Re-run the cells below to rebuild the index

# ── Uncomment to rebuild with new documents ──
# docs        = load_documents(DATA_DIR)
# chunks      = split_documents(docs)
# vectorstore = build_vectorstore(chunks, VECTORSTORE_DIR)
# rag_chain   = build_rag_chain(vectorstore, llm)
# print("✅ Index rebuilt with new documents!")

print("💡 Uncomment the lines above after adding new PDFs to data/")

## Step 14 — Save RAG Pipeline Module for Streamlit

In [ ]:
rag_module = '''
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

DATA_DIR        = Path("data")
VECTORSTORE_DIR = Path("vectorstore")
EMBED_MODEL     = "models/embedding-001"
LLM_MODEL       = "gemini-1.5-flash"
CHUNK_SIZE      = 512
CHUNK_OVERLAP   = 64
TOP_K           = 5

CLINICAL_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert clinical decision support assistant.
Answer ONLY from the provided context. If not found, say so clearly.
Never fabricate drug dosages, diagnoses, or treatment protocols.

Context:
{context}

Clinical Question:
{question}

Evidence-Based Clinical Answer:"""
)

def get_embeddings(api_key):
    return GoogleGenerativeAIEmbeddings(model=EMBED_MODEL, google_api_key=api_key)

def load_documents(data_dir):
    docs = []
    for p in data_dir.glob("**/*.pdf"):
        docs.extend(PyPDFLoader(str(p)).load())
    for p in data_dir.glob("**/*.txt"):
        docs.extend(TextLoader(str(p), encoding="utf-8").load())
    return docs

def split_documents(docs):
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    ).split_documents(docs)

def build_vectorstore(chunks, save_dir, api_key):
    vs = FAISS.from_documents(chunks, get_embeddings(api_key))
    vs.save_local(str(save_dir))
    return vs

def load_vectorstore(save_dir, api_key):
    return FAISS.load_local(str(save_dir), get_embeddings(api_key),
                            allow_dangerous_deserialization=True)

def get_llm(api_key):
    return ChatGoogleGenerativeAI(
        model=LLM_MODEL, google_api_key=api_key,
        temperature=0, convert_system_message_to_human=True
    )

def build_rag_chain(vs, llm):
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K}),
        return_source_documents=True,
        chain_type_kwargs={"prompt": CLINICAL_PROMPT},
    )

def run_query(chain, question):
    result = chain({"query": question})
    sources = [
        {"source": Path(d.metadata.get("source","unknown")).name,
         "page": d.metadata.get("page", "N/A"),
         "excerpt": d.page_content[:250]}
        for d in result["source_documents"]
    ]
    return result["result"], sources
'''

with open("rag_core.py", "w") as f:
    f.write(rag_module)

print("✅ rag_core.py saved — ready for Streamlit!")

---
## ✅ You're Done!

**Next steps:**
1. Add your own medical PDFs to the `data/` folder
2. Run `streamlit run app.py` for the web UI
3. Deploy to Streamlit Cloud (see README.md)

**RAG Pipeline Summary:**
| Component | Choice |
|---|---|
| LLM | Gemini 1.5 Flash |
| Embeddings | Google `embedding-001` |
| Vector Store | FAISS (local) |
| Framework | LangChain |
| Chunk Size | 512 characters |
| Top-K Retrieval | 5 chunks |